# 🚑 Smart Intersection Emergency Vehicle Detector (YOLO Training)

This Google Colab notebook automates the training of **YOLO11n / YOLOv8n** on merged emergency vehicle datasets.

### Target Classes:
- `0: ambulance`
- `1: fire_truck`
- `2: police_car`
- `3: vehicle` (normal traffic negative baseline)

Follow these steps to produce `best.pt` for the project.

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow pyyaml matplotlib seaborn

## 2. Configure Roboflow & Download Public Datasets

In [ ]:
import os
from roboflow import Roboflow

# Enter your Roboflow API Key
ROBOFLOW_API_KEY = "7tZbmCeZMVwZ0MhCX3aO"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

os.makedirs("raw_data", exist_ok=True)

print("Downloading Dataset 1: emergency-vehicle-ff3rh...")
d1 = rf.workspace("testing-iyut3").project("emergency-vehicle-ff3rh").version(3).download("yolov8", location="raw_data/d1")

print("Downloading Dataset 2: emergency-rg2sy...")
d2 = rf.workspace("wogud").project("emergency-rg2sy").version(3).download("yolov8", location="raw_data/d2")

print("Downloading Dataset 3: emergency-vehcile-detection...")
d3 = rf.workspace("pixel-pirates").project("emergency-vehcile-detection").version(1).download("yolov8", location="raw_data/d3")

print("All datasets downloaded!")

## 3. Merge Datasets & Remap Classes to Standard Contract

In [ ]:
import hashlib
import random
import shutil
from pathlib import Path
import yaml

TARGET_CLASSES = ["ambulance", "fire_truck", "police_car", "vehicle"]

CLASS_SYNONYMS = {
    "ambulance": 0,
    "firetruck": 1,
    "fire_truck": 1,
    "fire truck": 1,
    "police": 2,
    "police_car": 2,
    "police car": 2,
    "car": 3,
    "truck": 3,
    "vehicle": 3,
    "normal vehicle": 3,
    "bus": 3,
}

def compute_hash(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def load_class_map(src_dir: Path) -> dict:
    yamls = list(src_dir.glob("data.yaml")) + list(src_dir.glob("*.yaml"))
    if not yamls:
        return {}
    with open(yamls[0], "r", encoding="utf-8") as f:
        meta = yaml.safe_load(f)
    raw = meta.get("names", {})
    src_names = {i: n for i, n in enumerate(raw)} if isinstance(raw, list) else {int(k): v for k, v in raw.items()}
    mapping = {}
    for src_id, name in src_names.items():
        norm = str(name).strip().lower().replace("-", "_")
        mapping[src_id] = CLASS_SYNONYMS.get(norm)
    return mapping

def remap_and_save(src_lbl: Path, dest_lbl: Path, cmap: dict) -> bool:
    lines = []
    with open(src_lbl, "r", encoding="utf-8") as f:
        for l in f:
            parts = l.strip().split()
            if not parts:
                continue
            orig_cls = int(parts[0])
            tgt_cls = cmap.get(orig_cls) if cmap else (orig_cls if orig_cls < len(TARGET_CLASSES) else None)
            if tgt_cls is not None:
                lines.append(f"{tgt_cls} " + " ".join(parts[1:]) + "\n")
    dest_lbl.parent.mkdir(parents=True, exist_ok=True)
    with open(dest_lbl, "w", encoding="utf-8") as f:
        f.writelines(lines)
    return len(lines) > 0

output_dir = Path("processed_dataset")
for split in ["train", "val", "test"]:
    (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

source_dirs = [Path("raw_data/d1"), Path("raw_data/d2"), Path("raw_data/d3")]
pairs = []
seen = set()

for s in source_dirs:
    if not s.exists():
        continue
    cmap = load_class_map(s)
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        for img in s.rglob(ext):
            lbl = img.with_suffix(".txt")
            if not lbl.exists():
                alt = img.parents[1] / "labels" / f"{img.stem}.txt"
                if alt.exists():
                    lbl = alt
                else:
                    continue
            h = compute_hash(img)
            if h in seen:
                continue
            seen.add(h)
            pairs.append((img, lbl, cmap))

print(f"Total unique image-label pairs: {len(pairs)}")
random.seed(42)
random.shuffle(pairs)

n = len(pairs)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

splits = {
    "train": pairs[:n_train],
    "val": pairs[n_train:n_train+n_val],
    "test": pairs[n_train+n_val:]
}

for sname, items in splits.items():
    for img, lbl, cmap in items:
        dest_img = output_dir / "images" / sname / img.name
        dest_lbl = output_dir / "labels" / sname / f"{img.stem}.txt"
        shutil.copy2(img, dest_img)
        remap_and_save(lbl, dest_lbl, cmap)

dataset_yaml = {
    "path": str(output_dir.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(TARGET_CLASSES)}
}

with open(output_dir / "data.yaml", "w") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False)

print("Merged dataset ready at /content/processed_dataset/data.yaml")

## 4. Train YOLO (YOLO11n or YOLOv8n)

In [ ]:
from ultralytics import YOLO

# Choose model architecture: yolo11n.pt or yolov8n.pt
model = YOLO("yolo11n.pt")

results = model.train(
    data="processed_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=10,
    save=True,
    project="runs_traffic",
    name="emergency_yolo11n"
)

## 5. Evaluate Metrics (mAP50, Precision, Recall)

In [ ]:
metrics = model.val(data="processed_dataset/data.yaml", split="test")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

## 6. Download `best.pt`

In [ ]:
from google.colab import files

best_weights = "runs_traffic/emergency_yolo11n/weights/best.pt"
if os.path.exists(best_weights):
    print("Downloading best.pt to your computer...")
    files.download(best_weights)
else:
    print(f"Weights not found at {best_weights}. Check runs_traffic/")